# BDB27 — the 30–45° turn question

**This notebook is the whole interface.** Set `MODE` in §1 and press **Run All**. No terminal,
no commands to type.

**The project in one line** (`TASK.md` @ `f8ff492`). Does the Combine test the turn the NFL
asks defensive backs to execute — the shallow break at moderate speed? Measure the turns each
position actually makes in games, compare that with what the Combine tests, and report the gap
by turn band. Per-player coverage and the Cut Card are appendix-only.

**Provenance — read this before any number.**

* `MODE = "full"` is the final analysis. Every stage artefact is stamped `FULL`; only then is a
  number a result.
* `MODE = "sample"` is a plumbing run on this host's truncated tracking sample. Every artefact
  is stamped `PLACEHOLDER_SAMPLE: UNVALIDATED SAMPLE OUTPUT`. The headline prints a loud
  `UNVALIDATED SAMPLE OUTPUT — NOT A RESULT` banner, and nothing here is interpreted, reported
  or tuned against those numbers.

**What this notebook does, in order:** runs the seven stages of `TASK.md` in the spec's order
(`00_thresholds → 01_audit → 02_turns → 03_envelope → 04_demand_coverage → 05_validate →
06_figures`), showing each stage's exit code, wall time and status; then renders the headline
from the audit artefacts and displays whatever figures exist.

**What it never does:** compute a number of its own, substitute sample data for missing full
data, or count a stub as a pass.


## 1. Configuration — the only cell to edit

* `MODE="sample"` runs the chain on the data this host has. `MODE="full"` is the final run.
* `DATA_DIR=None` uses the repo's `data/raw`. Set it only when the files live elsewhere (the
  whole data directory, not a single file). An override writes a scratch config under
  `data/interim/tmp/` with absolutised paths, so its `config_sha` will **differ** from the
  committed evidence — use `None` for runs whose artefacts you intend to keep.
* `FORCE_STAGES` rebuilds only the named stages; `[]` reuses existing checkpoints.
* `CLEAN=True` is destructive but **inert unless `CLEAN_CONFIRM="DELETE"`**. Even then it only
  *moves* this chain's outputs and checkpoints to `data/interim/tmp/cleaned-<utc>/` — it never
  touches `data/raw`, and the move is reversible.
* `ONLY_STAGES` runs a subset (debugging); `[]` runs all seven in spec order.

There is nothing to edit below this cell.


In [ ]:
# ======================= CONFIG — the only cell to edit =======================
MODE          = "sample"   # "sample" (this host) | "full" (final run)
DATA_DIR      = None       # None = repo data/raw; else the data directory
FORCE_STAGES  = []         # e.g. ["00_thresholds", "01_audit"]; [] = reuse checkpoints
CLEAN         = False      # destructive; inert unless CLEAN_CONFIRM == "DELETE"
CLEAN_CONFIRM = ""         # safety interlock
ONLY_STAGES   = []         # e.g. ["02_turns"]; [] = all seven, in spec order
# =============================================================================

assert MODE in ("sample", "full"), MODE
assert DATA_DIR is None or isinstance(DATA_DIR, str), DATA_DIR
assert isinstance(FORCE_STAGES, (list, tuple)) and all(isinstance(s, str) and s for s in FORCE_STAGES), FORCE_STAGES
assert isinstance(ONLY_STAGES, (list, tuple)) and all(isinstance(s, str) and s for s in ONLY_STAGES), ONLY_STAGES
assert isinstance(CLEAN, bool) and isinstance(CLEAN_CONFIRM, str), (CLEAN, CLEAN_CONFIRM)

print(f"mode         : {MODE}")
print(f"data dir     : {DATA_DIR or 'repo default (data/raw)'}")
print(f"force stages : {list(FORCE_STAGES) or 'none — reuse checkpoints'}")
print(f"only stages  : {list(ONLY_STAGES) or 'all seven, in spec order'}")
print("clean        : "
      + ("ARMED — outputs and checkpoints will be moved aside (recoverable)"
         if CLEAN and CLEAN_CONFIRM == "DELETE"
         else ("IGNORED — set CLEAN_CONFIRM='DELETE' to confirm" if CLEAN else "off")))


## 2. Bootstrap — find the repo, resolve the data directory, list the inputs

Walks upward until it sees `config.yaml` and `src/run.py`, resolves `DATA_DIR`, and prints the
environment and a present/absent row for every input the current mode reads. A missing file is
always **named**; nothing is silently substituted.


In [ ]:
import copy
import subprocess
import sys
from pathlib import Path


def find_repo_root() -> Path:
    """Walk upward from the notebook's working directory."""
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "config.yaml").exists() and (parent / "src" / "run.py").exists():
            return parent
    raise RuntimeError(
        "Cannot locate the BDB27 repo root (need config.yaml + src/run.py). "
        "Open this notebook from inside the BDB27 checkout.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import yaml

BASE_CONFIG = yaml.safe_load((REPO / "config.yaml").read_text(encoding="utf-8"))
BLOCK = BASE_CONFIG["turn_coverage"]


def _resolve(value) -> Path:
    p = Path(value)
    return p if p.is_absolute() else REPO / p


USED_CONFIG = REPO / "config.yaml"
CONFIG_ARG = []
if DATA_DIR:
    override = Path(DATA_DIR).expanduser()
    override = (override if override.is_absolute() else (REPO / override)).resolve()
    if not override.is_dir():
        raise RuntimeError(f"DATA_DIR does not exist or is not a directory: {override}")
    scratch = copy.deepcopy(BASE_CONFIG)
    for key, value in scratch["turn_coverage"]["paths"].items():
        scratch["turn_coverage"]["paths"][key] = str(_resolve(value))
    scratch["turn_coverage"]["paths"]["raw"] = str(override)
    USED_CONFIG = REPO / "data" / "interim" / "tmp" / f"notebook_config_{MODE}.yaml"
    USED_CONFIG.parent.mkdir(parents=True, exist_ok=True)
    USED_CONFIG.write_text(yaml.safe_dump(scratch, sort_keys=False), encoding="utf-8")
    CONFIG_ARG = ["--config", str(USED_CONFIG)]
    BLOCK = scratch["turn_coverage"]

RAW_DIR = _resolve(BLOCK["paths"]["raw"])
OUTPUTS_DIR = _resolve(BLOCK["paths"]["outputs"])
AUDIT_DIR = _resolve(BLOCK["paths"]["audit"])
FIG_DIR = _resolve(BLOCK["paths"]["figures"])
CKPT_DIR = _resolve(BLOCK["paths"]["checkpoints"])
THRESHOLDS_JSON = _resolve(BLOCK["paths"]["thresholds"])
PREREG_MD = _resolve(BLOCK["paths"]["prereg"])
GATE_JSON = AUDIT_DIR / "gate.json"
AUDIT_CSV = AUDIT_DIR / "audit_items.csv"

FULL_GAME_FILE = BLOCK["files"]["game_tracking"]
FILE_TABLE = dict(BLOCK["files"])
if MODE == "sample":
    FILE_TABLE.update(BLOCK.get("sample_files", {}))

try:
    GIT_REV = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], cwd=str(REPO), text=True).strip()
except Exception:  # pragma: no cover - only outside a git checkout
    GIT_REV = "unknown"

print(f"python       : {sys.version.split()[0]}")
print(f"repo         : {REPO}")
print(f"git rev      : {GIT_REV}")
print(f"mode         : {MODE}")
print(f"config       : {USED_CONFIG}")
print(f"data dir     : {RAW_DIR}")
print(f"prereg       : {'present' if PREREG_MD.exists() else 'ABSENT — a full run is refused until it exists'}")
print("\ninputs the current mode reads")
for key, name in FILE_TABLE.items():
    path = RAW_DIR / name
    state = f"{path.stat().st_size / 1e6:,.1f} MB" if path.exists() else "ABSENT"
    print(f"  {key:<24} {name:<34} {state}")


## 3. The stage chain

The order is fixed by `TASK.md` (Layout and run order); it is not configurable here. Each stage
runs in this process through its own `main()`, and **every stage module and every `bdb27/`
helper is reloaded from disk before each call** — editing `src/` and re-running one cell is
enough to change behaviour.

| # | stage | what it is for | writes |
|---|---|---|---|
| 1 | `00_thresholds` | the thresholds contract (smoothing, floors, bins) | `outputs/thresholds.json` |
| 2 | `01_audit` | Phase 1 evidence, items a–m, and the gate | `outputs/audit/` |
| 3 | `02_turns` | kinematics and the per-turn table | turn table |
| 4 | `03_envelope` | per-player capacity envelope | envelope table |
| 5 | `04_demand_coverage` | game demand map and Turn Coverage | coverage tables |
| 6 | `05_validate` | transfer, feasibility, outcome tests | validation artefacts |
| 7 | `06_figures` | the deliverable figures | `outputs/figures/` |

Status vocabulary used below:

* **`OK — ran, artefacts written`** / **`OK — checkpoint reused, nothing rebuilt`** — a pass.
* **`NOT IMPLEMENTED (stub, exit 2)`** — the stage is a loud stub: it printed a refusal, wrote
  nothing, and is **never counted as a pass**.
* **`REFUSED (exit 2)`** — the stage declined honestly (missing input, wrong provenance). The
  reason is printed.
* **`FAILED (exit n)`** — a real failure; the captured output is printed.
* **`PENDING FULL RUN`** — only in `MODE="full"` when the full game tracking is absent. No stage
  is run and no sample file is substituted.


In [ ]:
import importlib
import importlib.util
import inspect
import sys
import time
import traceback
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path

#: The seven stages, in TASK.md order. Not configurable.
STAGE_ORDER = [
    "00_thresholds",
    "01_audit",
    "02_turns",
    "03_envelope",
    "04_demand_coverage",
    "05_validate",
    "06_figures",
]

#: The full tracking files Kaggle ships per season. Named in the preflight so a
#: missing file can never be confused with a sample run.
FULL_SEASON_FILES = (
    "game_tracking_2023.csv",
    "game_tracking_2024.csv",
    "game_tracking_2025.csv",
)

HELPER_PACKAGE = "bdb27"


def _purge_bytecode(module_name: str) -> None:
    """Delete the cached .pyc so a reload must recompile from source."""
    module = sys.modules.get(module_name)
    source = getattr(module, "__file__", None) if module is not None else None
    if not source or not str(source).endswith(".py"):
        return
    try:
        cached = importlib.util.cache_from_source(str(source))
    except (NotImplementedError, ValueError):
        return
    try:
        Path(cached).unlink()
    except OSError:
        pass


def reload_helpers() -> list:
    """Drop bdb27 and every submodule so the next import re-reads source from disk."""
    dropped = []
    for name in [n for n in list(sys.modules)
                 if n == HELPER_PACKAGE or n.startswith(HELPER_PACKAGE + ".")]:
        _purge_bytecode(name)
        sys.modules.pop(name, None)
        dropped.append(name)
    importlib.invalidate_caches()
    return dropped


def load_stage(module_name: str):
    """Import one stage module fresh from src/<module_name>.py."""
    reload_helpers()
    path = SRC / f"{module_name}.py"
    if not path.exists():
        raise FileNotFoundError(f"stage module absent: {path}")
    spec = importlib.util.spec_from_file_location(f"bdb27_nb_{module_name}", path)
    if spec is None or spec.loader is None:
        raise ImportError(f"cannot import {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


def artefact_snapshot() -> dict:
    """mtime+size of every file under outputs/ and the checkpoint dir."""
    seen = {}
    for root in (OUTPUTS_DIR, CKPT_DIR):
        if not root or not Path(root).exists():
            continue
        for path in Path(root).rglob("*"):
            if path.is_file():
                stat = path.stat()
                seen[str(path.relative_to(REPO))] = (stat.st_mtime_ns, stat.st_size)
    return seen


def run_stage(module_name: str, extra_args=()) -> dict:
    """Run one stage through its own main(). Never let an exception escape."""
    started = time.perf_counter()
    argv = ["--mode", MODE, *CONFIG_ARG, *list(extra_args)]
    buffer = StringIO()
    rc, error = 0, ""
    before = artefact_snapshot()
    old_argv = sys.argv
    try:
        module = load_stage(module_name)
        sys.argv = [f"{module_name}.py", *argv]
        with redirect_stdout(buffer), redirect_stderr(buffer):
            takes_argv = bool(inspect.signature(module.main).parameters)
            result = module.main(argv) if takes_argv else module.main()
        rc = 0 if result is None else int(result)
    except SystemExit as exc:
        rc = 0 if exc.code in (None, 0) else int(exc.code)
    except KeyboardInterrupt:
        raise
    except BaseException as exc:  # noqa: BLE001 - reported, never hidden
        if type(exc).__name__ == "StageRefused":
            rc, error = 2, f"REFUSED: {exc}"
        else:
            rc, error = 1, f"{type(exc).__name__}: {exc}"
        print(error, file=buffer)
        traceback.print_exc(file=buffer)
    finally:
        sys.argv = old_argv
    return {
        "stage": module_name,
        "rc": rc,
        "seconds": time.perf_counter() - started,
        "output": buffer.getvalue(),
        "error": error,
        "changed": artefact_snapshot() != before,
    }


def classify_outcome(rc, output_text, artefacts_changed: bool) -> dict:
    """Turn a stage's exit code and output into a status; only OK is a pass."""
    text = output_text or ""
    if rc == 0:
        reused = (not artefacts_changed) or ("exists (use --force)" in text)
        return {
            "status": ("OK — checkpoint reused, nothing rebuilt" if reused
                       else "OK — ran, artefacts written"),
            "passed": True,
            "stub": False,
            "reused": reused,
        }
    if "STAGE NOT IMPLEMENTED" in text:
        return {"status": f"NOT IMPLEMENTED (stub, exit {rc})",
                "passed": False, "stub": True, "reused": False}
    if "REFUSED" in text or str(text).startswith("REFUSED"):
        return {"status": f"REFUSED (exit {rc})", "passed": False, "stub": False, "reused": False}
    return {"status": f"FAILED (exit {rc})", "passed": False, "stub": False, "reused": False}


def full_tracking_preflight(raw_dir: Path, configured_name: str):
    """(blocked, message). A full run is blocked when its game tracking is absent."""
    configured = Path(raw_dir) / configured_name
    seasons = [Path(raw_dir) / name for name in FULL_SEASON_FILES]
    if configured.exists():
        return False, ""
    lines = ["PENDING FULL RUN — full-mode game tracking is absent; nothing is run.",
             f"  the pipeline reads : {configured}   (config.yaml files.game_tracking)"]
    for path in seasons:
        state = "present" if path.exists() else "MISSING"
        lines.append(f"  season file        : {path}   [{state}]")
    if all(path.exists() for path in seasons):
        lines.append("  the three season files exist, but config.yaml names a single file: "
                     "concatenate them to the configured filename (or change "
                     "files.game_tracking) before the full run. This notebook will not "
                     "substitute a layout it was not given.")
    lines.append("  sample data is NOT substituted. Set MODE=\"sample\" to exercise the "
                 "code paths on the sample instead.")
    return True, "\n".join(lines)


## 4. Run the chain

Runs the stages above in order, honouring `FORCE_STAGES` / `ONLY_STAGES` / `CLEAN`, and prints
one row per stage: exit code, wall time, and status. The table is the record — a stub row reads
`NOT IMPLEMENTED (stub, exit 2)` and the counts below never include it as a pass. A failing
stage does not stop the chain; the headline cell reports what actually exists.


In [ ]:
import shutil

if CLEAN and CLEAN_CONFIRM != "DELETE":
    print("CLEAN=True is IGNORED — set CLEAN_CONFIRM='DELETE' to confirm.")

if CLEAN and CLEAN_CONFIRM == "DELETE":
    holder = REPO / "data" / "interim" / "tmp" / ("cleaned-" + time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()))
    targets = [OUTPUTS_DIR / "audit", OUTPUTS_DIR / "figures",
               OUTPUTS_DIR / "thresholds.json", OUTPUTS_DIR / "run_log.csv",
               _resolve(BLOCK["paths"]["parquet"]), CKPT_DIR]
    print(f"CLEAN ARMED — moving generated artefacts into {holder} (recoverable)")
    for target in targets:
        if not target.exists():
            continue
        holder.mkdir(parents=True, exist_ok=True)
        destination = holder / target.name
        suffix = 1
        while destination.exists():
            suffix += 1
            destination = holder / f"{target.name}.{suffix}"
        shutil.move(str(target), str(destination))
        print(f"  moved {target} -> {destination}")
        if target.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            if target.name in ("audit", "figures"):
                (target / ".gitkeep").touch()
    print("  data/raw was not touched.")

stages = list(STAGE_ORDER)
if ONLY_STAGES:
    unknown = [s for s in ONLY_STAGES if s not in STAGE_ORDER]
    if unknown:
        print(f"ONLY_STAGES names unknown stages: {unknown} — ignored")
    stages = [s for s in stages if s in ONLY_STAGES]
    print(f"ONLY_STAGES restricts this run to: {stages}")

pending, pending_message = False, ""
if MODE == "full":
    pending, pending_message = full_tracking_preflight(RAW_DIR, FULL_GAME_FILE)
if pending:
    print(pending_message)

stage_results = []
print("\n" + "=" * 92)
print(f"RUN — mode={MODE}   config={USED_CONFIG}")
print("=" * 92)
for stage in stages:
    if pending:
        stage_results.append({
            "stage": stage, "rc": None, "seconds": 0.0, "output": "", "error": "",
            "changed": False,
            "outcome": {"status": "PENDING FULL RUN", "passed": False,
                        "stub": False, "reused": False},
        })
        continue
    args = ["--force"] if stage in FORCE_STAGES else []
    result = run_stage(stage, args)
    result["outcome"] = classify_outcome(result["rc"], result["output"], result["changed"])
    stage_results.append(result)
    print(f"\n--- {stage} ---")
    captured = result["output"].strip()
    if captured:
        for line in captured.splitlines()[-40:]:
            print(f"    {line}")
    if result["error"] and result["error"] not in captured:
        print(f"    {result['error']}")

print("\n" + "=" * 92)
print("STAGE RESULTS")
print("=" * 92)
for result in stage_results:
    exit_text = "—" if result["rc"] is None else f"exit {result['rc']}"
    elapsed = "—" if result["rc"] is None else f"{result['seconds']:.1f} s"
    print(f"  {result['stage']:<20} {exit_text:<8} {elapsed:<10} {result['outcome']['status']}")
print("-" * 92)
passed = [r for r in stage_results if r["outcome"]["passed"]]
stubs = [r for r in stage_results if r["outcome"]["stub"]]
failed = [r for r in stage_results
          if r["rc"] not in (None, 0) and not r["outcome"]["stub"]]
print(f"  passed (exit 0)  : {len(passed)} of {len(stage_results)}")
if stubs:
    print("  stubs, NOT passes: " + ", ".join(r["stage"] for r in stubs))
    print("                     (each printed NOT IMPLEMENTED, wrote nothing, exit 2)")
if failed:
    print("  failed           : " + ", ".join(
        f"{r['stage']} (exit {r['rc']})" for r in failed))
    print("                     captured output is printed above; the headline reads what exists")
if pending:
    print("  full-mode game tracking absent — every stage is PENDING FULL RUN; "
          "no sample file was substituted")


## 5. Figure gallery — every figure, its claim, its caveats

This section reads `outputs/figures/figures_manifest.json` and shows **every figure the stage
wrote**, in manifest order: title, claim, caveats, sources, the figure's own provenance stamp,
then the PNG displayed from its path. No figure is named in the notebook — add one in
`06_figures` and it appears here; if it is not in the manifest, it is not a figure this
notebook claims exists.

* A listed PNG that is **absent** prints `MISSING` against its manifest name — no placeholder
  image and no silent skip.
* **Sample provenance is stated once, in a banner at the top.** In `MODE="sample"` every data
  figure carries `PLACEHOLDER_SAMPLE` / `UNVALIDATED SAMPLE OUTPUT`: it exercises the drawing
  code and is **never presented as a result**.
* The proposal-stamped schematic (`status: PROPOSAL`) shows its **own PROPOSAL stamp**. It is a
  drawing of a proposed drill, not a data figure, and it is deliberately not sample-stamped —
  the gallery does not force one stamp onto every figure.
* When the manifest is absent, the cell says why, using the `06_figures` row from §4.


In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display

GALLERY_MANIFEST = FIG_DIR / "figures_manifest.json"
SAMPLE_TOKEN = "PLACEHOLDER_SAMPLE"


def figure_stage_note(results, stage="06_figures"):
    """Why there are no figures, in the words of this session's stage results."""
    row = next((r for r in (results or []) if r["stage"] == stage), None)
    if row is None:
        return "06_figures was not run in this session (ONLY_STAGES)"
    outcome = row["outcome"]
    if outcome["status"] == "PENDING FULL RUN":
        return "PENDING FULL RUN — 06_figures cannot run without the full game tracking"
    if outcome["stub"]:
        return f"06_figures did not run: {outcome['status']}; a stub writes no figure"
    if row["rc"] == 0:
        return "06_figures exited 0 but wrote no figure files"
    return f"06_figures did not finish: {outcome['status']}"


def resolve_figure_path(value, fig_dir):
    """A manifest path is repo-relative; fall back to the same file name in fig_dir."""
    if not value:
        return None
    path = Path(value)
    if not path.is_absolute():
        path = REPO / path
    if path.exists():
        return path
    fallback = Path(fig_dir) / Path(value).name
    return fallback if fallback.exists() else path


def figure_stamp(entry):
    """(the stamp this entry actually carries, whether it is the sample stamp)."""
    own = entry.get("status_stamp") or entry.get("stamp") or ""
    return own, SAMPLE_TOKEN in str(own)


def figure_gallery(manifest_path, fig_dir, mode, results=(), display_fn=display):
    """Print every figure the manifest lists, then display its PNG from disk.

    The manifest is the list of figures — none is named here. A listed PNG that is
    absent prints MISSING against the manifest's own name: no placeholder image and
    no silent skip. Sample provenance is stated once, in a banner at the top, and a
    sample figure is never presented as a result. A PROPOSAL-stamped schematic shows
    its own PROPOSAL stamp and is deliberately not given a sample badge.
    """
    manifest_path, fig_dir = Path(manifest_path), Path(fig_dir)
    print("=" * 92)
    print("FIGURE GALLERY — every figure the stage-06 manifest lists")
    print("=" * 92)
    print(f"  mode     : {mode}")
    print(f"  manifest : {manifest_path}")
    print(f"  images   : {fig_dir}   (displayed from path — nothing is embedded)")

    if not manifest_path.exists():
        print()
        print("  NOT RUN — figures manifest absent: nothing to list and nothing to display.")
        print(f"  Reason, from this session's stage results: {figure_stage_note(results)}")
        return {"figures": 0, "displayed": 0, "missing": [], "proposal": []}

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    entries = manifest.get("figures", {})
    names = list(entries)
    sample_names = [name for name in names if figure_stamp(entries[name])[1]]
    proposal_names = [name for name in names if entries[name].get("status") == "PROPOSAL"]

    print(f"  provenance: {manifest.get('provenance', 'ABSENT')}"
          f"   label: {manifest.get('label', 'ABSENT')}")
    print(f"  generated : {manifest.get('generated_utc', 'ABSENT')}"
          f"   config_sha: {manifest.get('config_sha', 'ABSENT')}")
    print(f"  figures   : {len(names)}")

    if sample_names:
        print()
        print("!" * 92)
        print(f"SAMPLE BANNER — {SAMPLE_TOKEN} ({manifest.get('label', 'UNVALIDATED SAMPLE OUTPUT')}):")
        print(f"  {len(sample_names)} of {len(names)} figures below are sample output.")
        print("  A sample figure is NOT A RESULT and is never presented as one: it exercises")
        print("  the drawing code on placeholder data and measures nothing about the 2027 class.")
        print("!" * 92)
    if proposal_names:
        print()
        print(f"PROPOSAL — {len(proposal_names)} figure(s) are proposals, not data figures: "
              + ", ".join(proposal_names))
        print("  A proposal carries its own PROPOSAL stamp; it is not sample-stamped, and it "
              "is not a finding.")
    if not sample_names and not proposal_names:
        print(f"  stamp    : {manifest.get('stamp', 'ABSENT')}")

    shown, missing = [], []
    for index, name in enumerate(names, start=1):
        entry = entries[name]
        print()
        print("-" * 92)
        print(f"  [{index}/{len(names)}] {name}")
        print(f"      title   : {entry.get('title', 'ABSENT')}")
        print(f"      claim   : {entry.get('claim', 'ABSENT')}")
        stamp, is_sample = figure_stamp(entry)
        if entry.get("status") == "PROPOSAL":
            print(f"      status  : PROPOSAL — {stamp or 'PROPOSED, no stamp'}")
            if entry.get("provenance_applied") is False:
                print("      stamp   : deliberately NOT sample-stamped — a schematic of a "
                      "proposed drill, not a data artefact")
        elif is_sample:
            print(f"      stamp   : {stamp}   (sample figure — NOT A RESULT)")
        else:
            print(f"      stamp   : {stamp or 'ABSENT'}")
        caveats = entry.get("caveats") or []
        for number, caveat in enumerate(caveats, start=1):
            print(f"      caveat {number}: {caveat}")
        if not caveats:
            print("      caveat  : ABSENT — the manifest carries no caveat for this figure")
        sources = entry.get("sources") or []
        print(f"      sources : {', '.join(sources) if sources else 'ABSENT'}")

        png = resolve_figure_path(entry.get("png"), fig_dir)
        if png is None:
            print(f"      picture : MISSING — {name} (the manifest gives no png path)")
            missing.append(name)
        elif not png.exists():
            print(f"      picture : MISSING — {name} (expected {entry.get('png')})")
            missing.append(name)
        else:
            print(f"      picture : {png.name}   {png.stat().st_size / 1024:.1f} KB")
            display_fn(Image(filename=str(png)))
            shown.append(name)

    claimed = {Path(e["png"]).name for e in entries.values() if e.get("png")}
    unclaimed = sorted(
        p.name for p in fig_dir.glob("*")
        if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".svg") and p.name not in claimed
    )
    if unclaimed:
        print()
        print("  on disk but NOT in the manifest (not displayed, not counted): "
              + ", ".join(unclaimed))

    print()
    print("=" * 92)
    print(f"GALLERY RESULT — {len(shown)} of {len(names)} figure(s) displayed, "
          f"{len(missing)} MISSING, {len(proposal_names)} PROPOSAL")
    if sample_names:
        print(f"  {len(sample_names)} of {len(names)} carry {SAMPLE_TOKEN}: sample output, "
              "NOT A RESULT.")
    if missing:
        print("  missing, by name: " + ", ".join(missing))
    print("=" * 92)
    return {"figures": len(names), "displayed": len(shown),
            "missing": missing, "proposal": proposal_names}


gallery = figure_gallery(GALLERY_MANIFEST, FIG_DIR, MODE, stage_results)


## 6. Headline — the block to send back

Everything here is read from `outputs/audit/gate.json` and `outputs/audit/audit_items.csv`. **No
number in this notebook is typed in.** It prints, in this order:

1. the **provenance banner taken from the artefact** — `PLACEHOLDER_SAMPLE` prints
   `UNVALIDATED SAMPLE OUTPUT — NOT A RESULT`; `FULL` prints `PROVENANCE: FULL`;
2. the gate **verdict**, why, the **primary group**, the **recommended smoother**, and the
   artefact's own timestamp and `config_sha`;
3. **conditions G1–G9**, each with its status, value, threshold and detail, including the
   conditions that are `PENDING FULL RUN` (those are not passes);
4. **audit items a–m**, each with its status and its key value (or, where the item records no
   single number, the evidence file it cites).

Anything the artefact does not contain prints `NOT RUN — <field> absent`; an absent gate prints
`NOT RUN — gate.json absent` and every downstream field follows suit. A missing item is named by
its letter.


In [ ]:
import csv
import json
import textwrap


def _load_json(path):
    p = Path(path)
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return None


def _load_items(path):
    p = Path(path)
    if not p.exists():
        return None
    with p.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def _expected_conditions():
    return [f"G{i}" for i in range(1, 10)]


def _gate_field(label, value):
    return str(value) if value not in (None, "") else f"NOT RUN — {label} absent"


def _wrapped(label, text, width=86):
    if text in (None, ""):
        print(f"{label:<22}: NOT RUN — {label.strip().lower()} absent")
        return
    print(f"{label:<22}:")
    for piece in textwrap.wrap(str(text), width=width):
        print(f"    {piece}")


def render_headline(gate_path, items_path):
    line = "=" * 92
    print(line)
    print("HEADLINE — read only from the audit artefacts".center(92))
    print(line)

    gate = _load_json(gate_path)
    items = _load_items(items_path)

    if gate is None:
        print(f"gate.json             : NOT RUN — gate.json absent ({gate_path})")
        for field in ("provenance", "verdict", "because", "primary_group",
                      "recommended_smoother"):
            print(f"{field:<22}: NOT RUN — {field} absent")
        print("\nGATE CONDITIONS")
        for cid in _expected_conditions():
            print(f"  {cid:<3}: NOT RUN — condition absent")
    else:
        provenance = gate.get("provenance")
        if provenance == "PLACEHOLDER_SAMPLE":
            print("  " + "UNVALIDATED SAMPLE OUTPUT — NOT A RESULT".center(88))
        elif provenance == "FULL":
            print("  " + "PROVENANCE: FULL".center(88))
        else:
            print(f"{'provenance':<22}: NOT RUN — provenance absent")

        print(f"{'generated_utc':<22}: {_gate_field('generated_utc', gate.get('generated_utc'))}")
        print(f"{'config_sha':<22}: {_gate_field('config_sha', gate.get('config_sha'))}")
        print(f"{'verdict':<22}: {_gate_field('verdict', gate.get('verdict'))}")
        print(f"{'primary group':<22}: {_gate_field('primary_group', gate.get('primary_group'))}")
        print(f"{'recommended smoother':<22}: "
              f"{_gate_field('recommended_smoother', gate.get('recommended_smoother'))}")
        _wrapped("because", gate.get("because"))

        conditions = {c.get("id"): c for c in (gate.get("conditions") or [])
                      if isinstance(c, dict)}
        print(f"\nGATE CONDITIONS ({len(conditions)} recorded)")
        for cid in _expected_conditions():
            condition = conditions.get(cid)
            if condition is None:
                print(f"  {cid:<3}: NOT RUN — condition absent")
                continue
            status = condition.get("status")
            print(f"  {cid:<3} {status if status not in (None, '') else 'NOT RUN — status absent'}")
            print(f"      condition : {_gate_field('condition', condition.get('condition'))}")
            print(f"      value     : {_gate_field('value', condition.get('value'))}")
            print(f"      threshold : {_gate_field('threshold', condition.get('threshold'))}")
            detail = condition.get("detail")
            if detail in (None, ""):
                print("      detail    : NOT RUN — detail absent")
            else:
                pieces = textwrap.wrap(str(detail), width=78)
                print(f"      detail    : {pieces[0]}")
                for piece in pieces[1:]:
                    print(f"                  {piece}")
        tally = {}
        for condition in conditions.values():
            key = str(condition.get("status"))
            tally[key] = tally.get(key, 0) + 1
        print("  tally     : " + ", ".join(f"{k}={v}" for k, v in sorted(tally.items())))

    print("\nAUDIT ITEMS a–m")
    if items is None:
        for letter in "abcdefghijklm":
            print(f"  {letter} : NOT RUN — item absent")
    else:
        by_letter = {str(row.get("item", "")).strip(): row for row in items}
        print(f"  recorded  : {len(items)} row(s); provenance "
              + ", ".join(sorted({(r.get('provenance') or 'NOT RUN — provenance absent')
                                  for r in items})))
        for letter in "abcdefghijklm":
            row = by_letter.get(letter)
            if row is None:
                print(f"  {letter} : NOT RUN — item absent")
                continue
            status = row.get("status") or "NOT RUN — status absent"
            value = (row.get("value") or "").strip()
            evidence = (row.get("evidence_file") or "").strip()
            key_value = value if value else (evidence or "NOT RUN — value absent")
            print(f"  {letter} : {status:<18} value/evidence: {key_value}")
            name = (row.get("name") or "").strip()
            if name:
                print(f"      {name}")
            finding = (row.get("finding") or "").strip()
            if finding:
                for piece in textwrap.wrap(finding, width=82):
                    print(f"      {piece}")

    print(line)


render_headline(GATE_JSON, AUDIT_CSV)


## 7. Phase 3 envelope — the variance route

Reads `outputs/envelope/manifest.json` and `outputs/envelope/variance_components.csv` and prints
the between-player variance route: the moment-matched `tau^2` and its status, the held-out
cross-check and whether the two routes agree, the `AT_BOUNDARY` count, and every fit that did
not converge with its reason. Nothing here is typed in. An absent artefact prints
`NOT RUN — envelope manifest absent`; an absent field prints `NOT RUN — <field> absent`.


In [ ]:
import csv
import json
from pathlib import Path


def _env_load_json(path):
    p = Path(path)
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return None


def _env_load_csv(path):
    p = Path(path)
    if not p.exists():
        return None
    with p.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def _env_field(label, value):
    return str(value) if value not in (None, "") else f"NOT RUN — {label} absent"


def render_envelope_headline(manifest_path, components_path):
    line = "=" * 92
    print(line)
    print("PHASE 3 ENVELOPE — THE VARIANCE ROUTE".center(92))
    print(line)

    manifest = _env_load_json(manifest_path)
    components = _env_load_csv(components_path)

    if manifest is None:
        print(f"envelope manifest      : NOT RUN — envelope manifest absent ({manifest_path})")
        for field in ("provenance", "artefact_version", "settings_fitted",
                      "players_fitted", "tau2_route", "crosscheck", "n_components",
                      "n_at_boundary", "n_disagreements", "convergence"):
            print(f"{field:<23}: NOT RUN — {field} absent")
        print(line)
        return

    provenance = manifest.get("provenance")
    if provenance == "PLACEHOLDER_SAMPLE":
        print("  " + "UNVALIDATED SAMPLE OUTPUT — NOT A RESULT".center(88))
    elif provenance == "FULL":
        print("  " + "PROVENANCE: FULL".center(88))
    else:
        print(f"{'provenance':<23}: NOT RUN — provenance absent")

    print(f"{'artefact_version':<23}: {_env_field('artefact_version', manifest.get('artefact_version'))}")
    settings = manifest.get("settings_fitted")
    print(f"{'settings_fitted':<23}: "
          + (", ".join(settings) if settings else "NOT RUN — settings_fitted absent"))
    players = manifest.get("players_fitted") or {}
    print(f"{'players_fitted':<23}: "
          + (", ".join(f"{k}={v}" for k, v in players.items())
             if players else "NOT RUN — players_fitted absent"))

    variance = manifest.get("variance") or {}
    route = variance.get("route")
    crosscheck = variance.get("crosscheck")
    print(f"{'tau2 route':<23}: "
          + (f"{route} (crosscheck: {crosscheck or 'NOT RUN — crosscheck absent'})"
             if route else "NOT RUN — tau2_route absent"))

    n_components = variance.get("n_components")
    n_boundary = variance.get("n_at_boundary")
    n_disagree = variance.get("n_disagreements")
    if n_components is not None:
        print(f"{'components':<23}: {n_components} component(s)")
    else:
        print(f"{'components':<23}: NOT RUN — n_components absent")
    print(f"{'AT_BOUNDARY':<23}: "
          + (f"{n_boundary} of {n_components} component(s) AT_BOUNDARY"
             if n_boundary is not None and n_components is not None
             else "NOT RUN — n_at_boundary absent"))
    if n_disagree is not None and n_components is not None:
        print(f"{'routes agree':<23}: {int(n_components) - int(n_disagree)}/{n_components} "
              f"(disagreements={n_disagree})")
    else:
        print(f"{'routes agree':<23}: NOT RUN — n_disagreements absent")

    if components is None:
        print(f"{'per component':<23}: NOT RUN — components absent")
    else:
        for row in components:
            setting = row.get("setting") or "NOT RUN — setting absent"
            group = row.get("position_group") or "NOT RUN — position_group absent"
            measure = row.get("measure") or "NOT RUN — measure absent"
            tau2 = row.get("tau2") or "NOT RUN — tau2 absent"
            status = row.get("tau2_status") or "NOT RUN — tau2_status absent"
            heldout = row.get("tau2_heldout_optimal") or ""
            agree = row.get("routes_agree") or ""
            extra = ""
            if heldout:
                extra += f" heldout_optimal={heldout}"
            if agree:
                extra += f" routes_agree={agree}"
            print(f"  {setting:<8} {group:<6} {measure:<16} tau2={tau2:<10} "
                  f"status={status:<19}{extra}")

    convergence = manifest.get("convergence") or {}
    n_fits = convergence.get("n_fits")
    n_converged = convergence.get("n_converged")
    n_bad = convergence.get("n_not_converged")
    if n_fits is not None and n_converged is not None:
        print(f"{'fits converged':<23}: {n_converged}/{n_fits} "
              f"(non-converged={n_bad})")
    else:
        print(f"{'convergence':<23}: NOT RUN — convergence absent")
    for explanation in convergence.get("explanations") or []:
        print("  non-converged          : "
              f"{explanation.get('setting')}/{explanation.get('position_group')}/"
              f"{explanation.get('model')} "
              f"reason={explanation.get('stop_reason')} "
              f"iterations={explanation.get('iterations')} "
              f"n_obs={explanation.get('n_obs')} "
              f"n_players={explanation.get('n_players')} "
              f"objective_gain={explanation.get('objective_gain_last')}")
    print(line)


render_envelope_headline(OUTPUTS_DIR / "envelope" / "manifest.json",
                         OUTPUTS_DIR / "envelope" / "variance_components.csv")


## 8. Phase 4 — demand map and the supply-vs-demand table

Reads `outputs/demand_coverage/manifest.json`, `coverage_players.parquet` and
`coverage_position_ranking.csv`, and prints the game demand mass, the coverage
distribution (combined, observed, extrapolated), the boundary/sharing statement,
the cut-gap summary, the outcome-weighting status and the illustrative Cut Card
count. **The demand map by turn band and position is the deliverable** under the
current focus (`TASK.md` @ `f8ff492`); per-player Turn Coverage and the Cut Card
are appendix-only, because the per-player variance sits at the boundary. Nothing
here is typed in; an absent manifest prints `NOT RUN — demand/coverage manifest
absent` and every field follows suit. **No Cut Card figure is drawn here** —
stage 04 writes data only and rendering belongs to stage 06.

In [ ]:
import csv
import json
from pathlib import Path


def _dc_load_json(path):
    p = Path(path)
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return None


def _dc_load_csv(path):
    p = Path(path)
    if not p.exists():
        return None
    with p.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def _dc_load_parquet(path):
    p = Path(path)
    if not p.exists():
        return None
    try:
        import pandas as pd
        return pd.read_parquet(p)
    except Exception:
        return None


def _dc_num(value, spec=".3f"):
    if value is None or value == "":
        return "n/a"
    try:
        return format(float(value), spec)
    except (TypeError, ValueError):
        return "n/a"


def render_demand_coverage_headline(manifest_path, coverage_path, ranking_path):
    line = "=" * 92
    print(line)
    print("PHASE 4 — DEMAND MAP AND TURN COVERAGE".center(92))
    print(line)

    manifest = _dc_load_json(manifest_path)
    if manifest is None:
        print(f"demand/coverage manifest : NOT RUN — demand/coverage manifest "
              f"absent ({manifest_path})")
        for field in ("provenance", "positions_covered", "demand_mass",
                      "coverage_distribution", "boundary_share", "cut_gaps",
                      "outcome_status", "cut_cards"):
            print(f"{field:<24}: NOT RUN — {field} absent")
        print(line)
        return

    provenance = manifest.get("provenance")
    if provenance == "PLACEHOLDER_SAMPLE":
        print("  " + "UNVALIDATED SAMPLE OUTPUT — NOT A RESULT".center(88))
    elif provenance == "FULL":
        print("  " + "PROVENANCE: FULL".center(88))
    else:
        print(f"{'provenance':<24}: NOT RUN — provenance absent")

    positions = manifest.get("positions") or []
    counts = manifest.get("demand_counts") or {}
    per_group = counts.get("per_group") or {}
    if positions:
        print("positions covered      : " + ", ".join(
            f"{g}={per_group.get(g, 0)} eligible turns" for g in positions))
    else:
        print("positions covered      : NOT RUN — positions_covered absent")

    if counts.get("total_demand_mass") is not None:
        print(f"demand mass            : "
              f"{_dc_num(counts.get('total_demand_mass'), '.0f')} weighted turns "
              f"(frequency); eligible={counts.get('eligible', 'n/a')} of "
              f"{counts.get('turns_in', 'n/a')} game turns")
    else:
        print("demand mass            : NOT RUN — demand_mass absent")

    coverage = manifest.get("coverage") or {}
    dist = coverage.get("distribution") or {}
    for scope in ("combined", "observed", "extrapolated"):
        values = dist.get(scope) or {}
        if values:
            print(f"coverage {scope:<13}: mean={_dc_num(values.get('mean'))} "
                  f"min={_dc_num(values.get('min'))} max={_dc_num(values.get('max'))}")
        else:
            print(f"coverage {scope:<13}: NOT RUN — coverage_distribution absent")

    if coverage.get("n_players") is not None:
        boundary = coverage.get("boundary") or {}
        print(f"boundary share         : {coverage.get('n_shared_players', 0)}/"
              f"{coverage.get('n_players', 0)} players in a shared group "
              f"({coverage.get('n_shared_groups', 0)}/"
              f"{coverage.get('n_groups_with_players', coverage.get('n_groups', 0))} "
              f"groups group-level); plant_retention "
              f"{boundary.get('n_components_at_boundary', 0)}/"
              f"{boundary.get('n_components', 0)} components AT_BOUNDARY; "
              f"all_group_level={coverage.get('all_group_level')}")
    else:
        print("boundary share         : NOT RUN — boundary_share absent")

    gaps = manifest.get("cut_gaps") or {}
    if gaps:
        print(f"cut gaps               : {gaps.get('n_players_with_gaps', 0)} player(s), "
              f"{gaps.get('n_player_cell_rows', 0)} player-cell row(s), "
              f"top_n={gaps.get('top_n', 'n/a')}, "
              f"mean gap={_dc_num(gaps.get('mean_gap_score'))}")
    else:
        print("cut gaps               : NOT RUN — cut_gaps absent")

    outcome = manifest.get("outcome_weighting") or {}
    if outcome.get("status"):
        detail = outcome.get("counts") or {}
        print(f"outcome weights        : {outcome.get('status')}"
              + (f" ({outcome.get('reason')})" if outcome.get("reason") else "")
              + f"; positions={','.join(outcome.get('positions') or [])}; "
                f"qualifying="
                f"{detail.get('qualifying_turns_in_scope', 0)} in scope")
    else:
        print("outcome weights        : NOT RUN — outcome_status absent")

    cards = manifest.get("cut_cards") or {}
    if cards:
        print(f"cut cards              : {cards.get('n_cards', 0)} selected "
              f"({cards.get('selection', 'n/a')}); rendering deferred to stage 06 "
              f"({cards.get('rendering', 'NOT RUN — cut_cards absent')})")
    else:
        print("cut cards              : NOT RUN — cut_cards absent")

    rows = _dc_load_parquet(coverage_path)
    if rows is None:
        print(f"{'coverage rows':<23}: NOT RUN — coverage rows absent")
    else:
        print(f"{'coverage rows':<23}: {len(rows)} Combine player(s), "
              f"interval_method={coverage.get('interval_method')}, "
              f"joint_bootstrap_draws={coverage.get('joint_bootstrap_draws')}")

    ranking = _dc_load_csv(ranking_path)
    if ranking is None:
        print(f"{'position ranking':<23}: NOT RUN — ranking absent")
    else:
        ordered = sorted(ranking, key=lambda r: float(r.get("rank_coverage_combined") or 1e9))
        if ordered:
            top = ordered[0]
            print(f"{'position ranking':<23}: {len(ranking)} row(s); lowest coverage "
                  f"{top.get('position_group')}/{top.get('role')} "
                  f"mean={_dc_num(top.get('mean_combined'))}")
        else:
            print(f"{'position ranking':<23}: 0 row(s)")
    print(line)


render_demand_coverage_headline(
    OUTPUTS_DIR / "demand_coverage" / "manifest.json",
    OUTPUTS_DIR / "demand_coverage" / "coverage_players.parquet",
    OUTPUTS_DIR / "demand_coverage" / "coverage_position_ranking.csv",
)


## 9. What is still missing

This notebook reports the pipeline as it stands; it does not pretend a stage exists.

* Phases 5–6 of `TASK.md` are being built stage by stage. Any stage that is not built runs as a
  **loud stub**: `NOT IMPLEMENTED`, exit 2, no artefact, no checkpoint — reported above and
  never counted as a pass.
* A **full run** needs the full game tracking named by `config.yaml` (Kaggle ships it per
  season). Until it is present, `MODE="full"` reports `PENDING FULL RUN` and runs nothing.
* `notes/prereg.md` does not exist yet. `TASK.md` requires it before the first full-data run;
  the gate records this as G9.
* The audit's own gate (`outputs/audit/gate.json`) is the authority on what may proceed —
  including its stop conditions. Read it in §6, not from memory.
